# Unidad IV: Pruebas de Hipótesis, Regresión y ANOVA - Práctica

## Introducción

En este notebook practicaremos:

* Pruebas de hipótesis para medias (Z y t)
* Pruebas de hipótesis para proporciones
* Regresión lineal simple
* Análisis de correlación
* ANOVA de un factor
* Interpretación de p-valores

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import seaborn as sns

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Bibliotecas importadas correctamente")

## Ejercicio 1: Prueba t para una Media

### Problema
Una empresa afirma que el tiempo promedio de entrega es de 3 días. Una muestra de 25 entregas tiene una media de 3.4 días con desviación estándar de 0.8 días.

**Hipótesis:**
* H₀: μ = 3 días (la afirmación es correcta)
* H₁: μ ≠ 3 días (la afirmación es incorrecta)

¿Hay evidencia para rechazar la afirmación al nivel α = 0.05?

In [0]:
# Datos
n = 25
xbar = 3.4
s = 0.8
mu_0 = 3  # valor bajo H₀
alpha = 0.05

print("=" * 70)
print("PRUEBA T PARA UNA MEDIA: Tiempo de entrega")
print("=" * 70)
print(f"\nDatos de la muestra:")
print(f"  n = {n}")
print(f"  x̄ = {xbar} días")
print(f"  s = {s} días")
print(f"\nHipótesis:")
print(f"  H₀: μ = {mu_0} días")
print(f"  H₁: μ ≠ {mu_0} días (prueba bilateral)")
print(f"  Nivel de significancia: α = {alpha}")

# Paso 1: Calcular estadístico de prueba
error_estandar = s / np.sqrt(n)
t_estadistico = (xbar - mu_0) / error_estandar

print(f"\nPaso 1: Calcular estadístico de prueba")
print(f"  Error estándar: s/√n = {s}/√{n} = {error_estandar:.4f}")
print(f"  t = (x̄ - μ₀) / (s/√n)")
print(f"  t = ({xbar} - {mu_0}) / {error_estandar:.4f}")
print(f"  t = {t_estadistico:.4f}")

# Paso 2: Calcular valor crítico
gl = n - 1
t_critico = stats.t.ppf(1 - alpha/2, gl)

print(f"\nPaso 2: Valor crítico")
print(f"  Grados de libertad: gl = n - 1 = {gl}")
print(f"  t crítico (α/2 = {alpha/2}): ±{t_critico:.4f}")
print(f"  Región de rechazo: t < -{t_critico:.4f} o t > {t_critico:.4f}")

# Paso 3: Calcular p-valor
p_valor = 2 * (1 - stats.t.cdf(abs(t_estadistico), gl))

print(f"\nPaso 3: Calcular p-valor")
print(f"  p-valor = 2 × P(T > |t|) = {p_valor:.4f}")

# Paso 4: Decisión
print(f"\nPaso 4: Decisión")
if abs(t_estadistico) > t_critico:
    decision = "Rechazar H₀"
    print(f"  |t| = {abs(t_estadistico):.4f} > {t_critico:.4f}")
    print(f"  Decisión: {decision}")
    print(f"  Conclusión: Hay evidencia suficiente para rechazar la afirmación.")
    print(f"  El tiempo promedio de entrega es significativamente diferente de 3 días.")
else:
    decision = "No rechazar H₀"
    print(f"  |t| = {abs(t_estadistico):.4f} ≤ {t_critico:.4f}")
    print(f"  Decisión: {decision}")
    print(f"  Conclusión: No hay evidencia suficiente para rechazar la afirmación.")

if p_valor < alpha:
    print(f"\n  Usando p-valor: {p_valor:.4f} < {alpha} → Rechazar H₀")
else:
    print(f"\n  Usando p-valor: {p_valor:.4f} ≥ {alpha} → No rechazar H₀")

# Verificación con scipy
print(f"\nVerificación con scipy.stats:")
# Simular datos para usar ttest_1samp
np.random.seed(42)
muestra_simulada = np.random.normal(xbar, s, n)
t_scipy, p_scipy = stats.ttest_1samp(muestra_simulada, mu_0)
print(f"  t = {t_scipy:.4f}, p-valor = {p_scipy:.4f}")

# Visualización
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Distribución t con región de rechazo
t_vals = np.linspace(-4, 4, 1000)
y_vals = stats.t.pdf(t_vals, gl)

ax1.plot(t_vals, y_vals, 'b-', linewidth=2, label=f't({gl})')
ax1.fill_between(t_vals, 0, y_vals, where=(t_vals < -t_critico), 
                 color='red', alpha=0.3, label='Región de rechazo')
ax1.fill_between(t_vals, 0, y_vals, where=(t_vals > t_critico), 
                 color='red', alpha=0.3)
ax1.axvline(t_estadistico, color='green', linestyle='--', linewidth=2, 
            label=f't observado = {t_estadistico:.3f}')
ax1.axvline(-t_critico, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.axvline(t_critico, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_xlabel('Valor t')
ax1.set_ylabel('Densidad')
ax1.set_title(f'Distribución t con gl={gl} y α={alpha}')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfico 2: Intervalo de confianza vs valor H₀
ci_lower = xbar - t_critico * error_estandar
ci_upper = xbar + t_critico * error_estandar

ax2.errorbar(1, xbar, yerr=t_critico*error_estandar, fmt='o', 
             markersize=10, capsize=10, capthick=2, color='blue',
             label=f'IC 95%: [{ci_lower:.2f}, {ci_upper:.2f}]')
ax2.axhline(mu_0, color='red', linestyle='--', linewidth=2, 
            label=f'H₀: μ = {mu_0}')
ax2.axhline(xbar, color='green', linestyle='--', linewidth=1, 
            alpha=0.5, label=f'x̄ = {xbar}')
ax2.set_xlim(0.5, 1.5)
ax2.set_ylabel('Tiempo (días)')
ax2.set_title('Media Muestral con IC 95% vs H₀')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
ax2.set_xticks([])

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("INTERPRETACIÓN:")
print("=" * 70)
print(f"Si p-valor < α: evidencia fuerte contra H₀")
print(f"Si p-valor ≥ α: no hay evidencia suficiente contra H₀")
print(f"\nEn este caso: p-valor = {p_valor:.4f}, α = {alpha}")
if p_valor < alpha:
    print(f"Hay evidencia estadísticamente significativa.")
else:
    print(f"No hay evidencia estadísticamente significativa.")

## Ejercicio 2: Regresión Lineal Simple

### Problema
Analizar la relación entre gastos en publicidad (X) y ventas (Y).

**Objetivos:**
1. Ajustar modelo de regresión lineal: Y = β₀ + β₁X
2. Interpretar coeficientes
3. Calcular R² (coeficiente de determinación)
4. Hacer predicciones

In [0]:
# Datos: Publicidad (miles $) vs Ventas (miles $)
np.random.seed(42)
publicidad = np.array([10, 15, 18, 22, 25, 30, 35, 40, 45, 50, 55, 60])
ventas = 50 + 2.5 * publicidad + np.random.normal(0, 8, len(publicidad))

print("=" * 70)
print("REGRESIÓN LINEAL SIMPLE: Publicidad vs Ventas")
print("=" * 70)
print(f"\nDatos: {len(publicidad)} observaciones")
print(f"Variable independiente (X): Gasto en publicidad (miles $)")
print(f"Variable dependiente (Y): Ventas (miles $)\n")

# Crear DataFrame
df = pd.DataFrame({'Publicidad': publicidad, 'Ventas': ventas})
print(df.to_string(index=False))

# Estadísticas descriptivas
print(f"\nEstadísticas descriptivas:")
print(f"  Publicidad: media = ${publicidad.mean():.2f}k, std = ${publicidad.std():.2f}k")
print(f"  Ventas: media = ${ventas.mean():.2f}k, std = ${ventas.std():.2f}k")

# Calcular regresión lineal
slope, intercept, r_value, p_value, std_err = stats.linregress(publicidad, ventas)

print(f"\n" + "=" * 70)
print("MODELO DE REGRESIÓN:")
print("=" * 70)
print(f"\nY = β₀ + β₁X")
print(f"\nCoeficientes:")
print(f"  β₀ (Intercepto) = {intercept:.4f}")
print(f"     Interpretación: Ventas base cuando publicidad = $0")
print(f"  β₁ (Pendiente) = {slope:.4f}")
print(f"     Interpretación: Por cada $1k adicional en publicidad,")
print(f"     las ventas aumentan ${slope:.2f}k en promedio")

print(f"\nModelo ajustado:")
print(f"  Ventas = {intercept:.2f} + {slope:.2f} × Publicidad")

# Coeficiente de determinación
r_cuadrado = r_value ** 2
print(f"\nBondad de ajuste:")
print(f"  R² = {r_cuadrado:.4f} ({r_cuadrado*100:.2f}%)")
print(f"  Interpretación: El {r_cuadrado*100:.1f}% de la variabilidad en Ventas")
print(f"  es explicada por Publicidad")

# Correlación
correlacion = r_value
print(f"\n  Correlación (r): {correlacion:.4f}")
if correlacion > 0.7:
    fuerza = "fuerte"
elif correlacion > 0.4:
    fuerza = "moderada"
else:
    fuerza = "débil"
print(f"  Relación lineal {fuerza} y positiva")

# Prueba de hipótesis para la pendiente
print(f"\nPrueba de hipótesis:")
print(f"  H₀: β₁ = 0 (no hay relación lineal)")
print(f"  H₁: β₁ ≠ 0 (hay relación lineal)")
print(f"  p-valor = {p_value:.6f}")
if p_value < 0.05:
    print(f"  Decisión: Rechazar H₀ (p < 0.05)")
    print(f"  La relación es estadísticamente significativa")
else:
    print(f"  Decisión: No rechazar H₀ (p ≥ 0.05)")

# Predicciones
print(f"\n" + "=" * 70)
print("PREDICCIONES:")
print("=" * 70)
valores_prediccion = [20, 35, 50]
for x_pred in valores_prediccion:
    y_pred = intercept + slope * x_pred
    print(f"\nSi Publicidad = ${x_pred}k:")
    print(f"  Ventas predichas = {intercept:.2f} + {slope:.2f} × {x_pred}")
    print(f"  Ventas predichas = ${y_pred:.2f}k")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Diagrama de dispersión con línea de regresión
ax1 = axes[0, 0]
ax1.scatter(publicidad, ventas, s=80, alpha=0.7, edgecolors='black', 
            label='Datos observados')
x_line = np.linspace(publicidad.min(), publicidad.max(), 100)
y_line = intercept + slope * x_line
ax1.plot(x_line, y_line, 'r-', linewidth=2, 
         label=f'Y = {intercept:.2f} + {slope:.2f}X')
ax1.set_xlabel('Publicidad (miles $)')
ax1.set_ylabel('Ventas (miles $)')
ax1.set_title(f'Regresión Lineal (R² = {r_cuadrado:.3f})')
ax1.legend()
ax1.grid(alpha=0.3)

# Gráfico 2: Residuos
ax2 = axes[0, 1]
y_pred = intercept + slope * publicidad
residuos = ventas - y_pred
ax2.scatter(y_pred, residuos, s=80, alpha=0.7, edgecolors='black')
ax2.axhline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Valores predichos')
ax2.set_ylabel('Residuos')
ax2.set_title('Gráfico de Residuos')
ax2.grid(alpha=0.3)

# Gráfico 3: Q-Q plot de residuos
ax3 = axes[1, 0]
stats.probplot(residuos, dist="norm", plot=ax3)
ax3.set_title('Q-Q Plot de Residuos (normalidad)')
ax3.grid(alpha=0.3)

# Gráfico 4: Valores observados vs predichos
ax4 = axes[1, 1]
ax4.scatter(ventas, y_pred, s=80, alpha=0.7, edgecolors='black', 
            label='Datos')
min_val = min(ventas.min(), y_pred.min())
max_val = max(ventas.max(), y_pred.max())
ax4.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, 
         label='Línea perfecta')
ax4.set_xlabel('Ventas Observadas')
ax4.set_ylabel('Ventas Predichas')
ax4.set_title('Observado vs Predicho')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla de resultados
df_resultados = pd.DataFrame({
    'Publicidad': publicidad,
    'Ventas': ventas,
    'Predicción': y_pred,
    'Residuo': residuos
})
print(f"\nTabla de resultados:")
print(df_resultados.to_string(index=False))

print(f"\nError cuadrático medio (MSE): {(residuos**2).mean():.2f}")
print(f"Raíz del MSE (RMSE): {np.sqrt((residuos**2).mean()):.2f}")

## Ejercicio 3: ANOVA de Un Factor

### Problema
Comparar las ventas promedio de 3 estrategias de marketing diferentes.

**Hipótesis:**
* H₀: μ₁ = μ₂ = μ₃ (las medias son iguales)
* H₁: Al menos una media es diferente

¿Hay diferencia significativa entre las estrategias al nivel α = 0.05?

In [0]:
# Datos: Ventas (miles $) por estrategia
np.random.seed(42)
estrategia_A = np.random.normal(80, 10, 15)
estrategia_B = np.random.normal(90, 12, 15)
estrategia_C = np.random.normal(85, 11, 15)

print("=" * 70)
print("ANOVA DE UN FACTOR: Comparación de estrategias de marketing")
print("=" * 70)
print(f"\nDatos: 3 estrategias, 15 observaciones cada una\n")

# Estadísticas por grupo
grupos = {
    'Estrategia A': estrategia_A,
    'Estrategia B': estrategia_B,
    'Estrategia C': estrategia_C
}

print("Estadísticas descriptivas:")
print("="*70)
for nombre, datos in grupos.items():
    print(f"{nombre}:")
    print(f"  n = {len(datos)}")
    print(f"  Media = ${datos.mean():.2f}k")
    print(f"  Desv. Est. = ${datos.std():.2f}k")
    print(f"  Mínimo = ${datos.min():.2f}k")
    print(f"  Máximo = ${datos.max():.2f}k")
    print()

# Realizar ANOVA
F_estadistico, p_valor = stats.f_oneway(estrategia_A, estrategia_B, estrategia_C)

print("\n" + "=" * 70)
print("PRUEBA ANOVA:")
print("=" * 70)
print(f"\nHipótesis:")
print(f"  H₀: μ₁ = μ₂ = μ₃ (todas las medias son iguales)")
print(f"  H₁: Al menos una media es diferente")
print(f"  Nivel de significancia: α = 0.05")

print(f"\nResultados:")
print(f"  Estadístico F = {F_estadistico:.4f}")
print(f"  p-valor = {p_valor:.6f}")

# Decisión
print(f"\nDecisión:")
if p_valor < 0.05:
    print(f"  p-valor ({p_valor:.6f}) < 0.05")
    print(f"  Rechazar H₀")
    print(f"  Conclusión: Hay diferencia significativa entre al menos dos estrategias")
else:
    print(f"  p-valor ({p_valor:.6f}) ≥ 0.05")
    print(f"  No rechazar H₀")
    print(f"  Conclusión: No hay evidencia de diferencia significativa")

# Cálculos detallados de ANOVA
print(f"\n" + "=" * 70)
print("TABLA ANOVA (cálculos detallados):")
print("=" * 70)

# Combinar todos los datos
todos_datos = np.concatenate([estrategia_A, estrategia_B, estrategia_C])
media_total = todos_datos.mean()

# Número de grupos y observaciones
k = 3  # número de grupos
n_total = len(todos_datos)

# Suma de cuadrados entre grupos (SSB)
SSB = sum([len(grupo) * (grupo.mean() - media_total)**2 
           for grupo in [estrategia_A, estrategia_B, estrategia_C]])

# Suma de cuadrados dentro de grupos (SSW)
SSW = sum([sum((x - grupo.mean())**2) 
           for x, grupo in zip(
               [estrategia_A, estrategia_B, estrategia_C],
               [estrategia_A, estrategia_B, estrategia_C]
           )])

# Suma de cuadrados total (SST)
SST = sum((todos_datos - media_total)**2)

# Grados de libertad
gl_entre = k - 1
gl_dentro = n_total - k
gl_total = n_total - 1

# Cuadrados medios
MSB = SSB / gl_entre
MSW = SSW / gl_dentro

# Estadístico F
F_calc = MSB / MSW

print(f"\nFuente de variación:")
print(f"\n1. ENTRE GRUPOS (Between):")
print(f"   Suma de cuadrados (SSB): {SSB:.2f}")
print(f"   Grados de libertad: {gl_entre}")
print(f"   Cuadrado medio (MSB): {MSB:.2f}")

print(f"\n2. DENTRO DE GRUPOS (Within):")
print(f"   Suma de cuadrados (SSW): {SSW:.2f}")
print(f"   Grados de libertad: {gl_dentro}")
print(f"   Cuadrado medio (MSW): {MSW:.2f}")

print(f"\n3. TOTAL:")
print(f"   Suma de cuadrados (SST): {SST:.2f}")
print(f"   Grados de libertad: {gl_total}")
print(f"   Verificación: SSB + SSW = {SSB + SSW:.2f} ≈ SST = {SST:.2f}")

print(f"\n4. ESTADÍSTICO F:")
print(f"   F = MSB / MSW = {MSB:.2f} / {MSW:.2f} = {F_calc:.4f}")

# Visualización
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Boxplots
ax1 = axes[0, 0]
data_plot = [estrategia_A, estrategia_B, estrategia_C]
box_parts = ax1.boxplot(data_plot, labels=['Estrategia A', 'Estrategia B', 'Estrategia C'],
                         patch_artist=True)
for patch, color in zip(box_parts['boxes'], ['lightblue', 'lightgreen', 'lightcoral']):
    patch.set_facecolor(color)
ax1.axhline(media_total, color='red', linestyle='--', linewidth=2, 
            label=f'Media total = ${media_total:.2f}k')
ax1.set_ylabel('Ventas (miles $)')
ax1.set_title('Comparación de Estrategias')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Gráfico 2: Medias con IC 95%
ax2 = axes[0, 1]
medias = [estrategia_A.mean(), estrategia_B.mean(), estrategia_C.mean()]
error_est = [estrategia_A.std()/np.sqrt(len(estrategia_A)),
             estrategia_B.std()/np.sqrt(len(estrategia_B)),
             estrategia_C.std()/np.sqrt(len(estrategia_C))]

x_pos = np.arange(len(medias))
ax2.bar(x_pos, medias, color=['lightblue', 'lightgreen', 'lightcoral'], 
        alpha=0.7, edgecolor='black')
ax2.errorbar(x_pos, medias, yerr=[1.96*e for e in error_est], 
             fmt='none', color='black', capsize=10, capthick=2)
ax2.axhline(media_total, color='red', linestyle='--', linewidth=2, 
            label='Media total')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['Estrategia A', 'Estrategia B', 'Estrategia C'])
ax2.set_ylabel('Ventas promedio (miles $)')
ax2.set_title('Medias con IC 95%')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# Gráfico 3: Distribución F
ax3 = axes[1, 0]
f_vals = np.linspace(0, 6, 1000)
y_vals = stats.f.pdf(f_vals, gl_entre, gl_dentro)
ax3.plot(f_vals, y_vals, 'b-', linewidth=2, label=f'F({gl_entre}, {gl_dentro})')

# Valor crítico
F_critico = stats.f.ppf(0.95, gl_entre, gl_dentro)
ax3.fill_between(f_vals, 0, y_vals, where=(f_vals > F_critico), 
                 color='red', alpha=0.3, label='Región de rechazo')
ax3.axvline(F_estadistico, color='green', linestyle='--', linewidth=2, 
            label=f'F observado = {F_estadistico:.3f}')
ax3.axvline(F_critico, color='red', linestyle='--', linewidth=1, alpha=0.7,
            label=f'F crítico = {F_critico:.3f}')
ax3.set_xlabel('Valor F')
ax3.set_ylabel('Densidad')
ax3.set_title(f'Distribución F con α=0.05')
ax3.legend()
ax3.grid(alpha=0.3)

# Gráfico 4: Datos individuales
ax4 = axes[1, 1]
for i, (nombre, datos) in enumerate(grupos.items()):
    x_vals = np.random.normal(i+1, 0.04, len(datos))
    ax4.scatter(x_vals, datos, alpha=0.6, s=50)
    ax4.plot([i+0.8, i+1.2], [datos.mean(), datos.mean()], 'r-', linewidth=3)

ax4.axhline(media_total, color='red', linestyle='--', linewidth=2, 
            label='Media total')
ax4.set_xticks([1, 2, 3])
ax4.set_xticklabels(['Estrategia A', 'Estrategia B', 'Estrategia C'])
ax4.set_ylabel('Ventas (miles $)')
ax4.set_title('Datos Individuales por Estrategia')
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("INTERPRETACIÓN:")
print("=" * 70)
print(f"\nEl estadístico F mide la razón entre:")
print(f"  - Variabilidad ENTRE grupos (diferencias entre estrategias)")
print(f"  - Variabilidad DENTRO de grupos (variabilidad natural)")
print(f"\nF grande → Mayor evidencia de diferencias entre grupos")
print(f"\nEn este caso: F = {F_estadistico:.4f}, p = {p_valor:.6f}")
if p_valor < 0.05:
    print(f"Las estrategias tienen efectos significativamente diferentes.")
else:
    print(f"No hay evidencia suficiente de diferencias entre estrategias.")

## Resumen y Conclusiones

### Conceptos Clave

**Pruebas de Hipótesis:**
* H₀ (hipótesis nula): Lo que asumimos verdadero inicialmente
* H₁ (hipótesis alternativa): Lo que queremos probar
* p-valor: Probabilidad de observar los datos si H₀ es verdadera
* Regla: Si p-valor < α, rechazar H₀
* Error Tipo I (α): Rechazar H₀ cuando es verdadera
* Error Tipo II (β): No rechazar H₀ cuando es falsa

**Regresión Lineal:**
* Modelo: Y = β₀ + β₁X + ε
* β₀: Intercepto (valor de Y cuando X = 0)
* β₁: Pendiente (cambio en Y por unidad de X)
* R²: Proporción de variabilidad explicada (0 a 1)
* Correlación (r): Fuerza de la relación lineal (-1 a 1)

**ANOVA:**
* Compara medias de 3+ grupos
* H₀: Todas las medias son iguales
* Estadístico F = MSB / MSW
* MSB: Variabilidad entre grupos
* MSW: Variabilidad dentro de grupos
* F grande → Evidencia de diferencias

### Fórmulas Principales

**Prueba t para una media:**
```
t = (x̄ - μ₀) / (s/√n)
gl = n - 1
```

**Regresión lineal:**
```
β₁ = Σ[(xᵢ-x̄)(yᵢ-ȳ)] / Σ(xᵢ-x̄)²
β₀ = ȳ - β₁x̄
R² = 1 - SSE/SST
r = Σ[(xᵢ-x̄)(yᵢ-ȳ)] / √[Σ(xᵢ-x̄)² × Σ(yᵢ-ȳ)²]
```

**ANOVA:**
```
F = MSB / MSW
MSB = SSB / (k-1)
MSW = SSW / (N-k)
SST = SSB + SSW
```

### Herramientas Python

```python
# Prueba t de una muestra
from scipy import stats
t_stat, p_value = stats.ttest_1samp(muestra, mu_0)

# Prueba t de dos muestras
t_stat, p_value = stats.ttest_ind(grupo1, grupo2)

# Regresión lineal
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
R_cuadrado = r_value ** 2

# Correlación de Pearson
r, p_value = stats.pearsonr(x, y)

# ANOVA
F_stat, p_value = stats.f_oneway(grupo1, grupo2, grupo3)
```

### Interpretación de p-valores

* **p < 0.01**: Evidencia muy fuerte contra H₀
* **p < 0.05**: Evidencia fuerte contra H₀ (umbral común)
* **p < 0.10**: Evidencia débil contra H₀
* **p ≥ 0.10**: No hay evidencia suficiente contra H₀

### Aplicaciones en Negocios

* **Pruebas de hipótesis**: Validar afirmaciones, comparar grupos
* **Regresión**: Predecir ventas, modelar relaciones
* **Correlación**: Identificar factores relacionados
* **ANOVA**: Comparar estrategias, evaluar tratamientos